In [14]:
import torch.nn as nn
import torch
import pandas as pd
import json
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score

In [15]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self,
        input_dim: int = 1,
        hidden_dim: int = 64,
        latent_dim: int = 20,
        num_layers: int = 1,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # ---------- Encoder ---------- #
        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Bottleneck ---------- #
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)

        # ---------- Decoder ---------- #
        self.decoder = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        # ---------- Output ---------- #
        self.output_layer = nn.Linear(hidden_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected (B,T,F), got {x.shape}")

        batch_size, seq_len, _ = x.shape
        _, (h, _) = self.encoder(x)
        h_last = h[-1]

        latent = self.to_latent(h_last)
        hidden_dec = self.from_latent(latent)

        dec_in = hidden_dec.unsqueeze(1).repeat(1, seq_len, 1)

        h0 = hidden_dec.unsqueeze(0).repeat(self.num_layers, 1, 1)
        c0 = torch.zeros_like(h0)

        out, _ = self.decoder(dec_in, (h0, c0))
        out = self.output_layer(out)

        return out

In [16]:
def load_series(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


def load_windows(json_path: str, file_key: str) -> list:
    with open(json_path, "r") as f:
        data = json.load(f)
    return data[file_key]


def build_labels(df: pd.DataFrame, windows: list) -> pd.DataFrame:
    labels = np.zeros(len(df))

    for start, end in windows:
        start = pd.to_datetime(start)
        end = pd.to_datetime(end)

        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        labels[mask] = 1

    df["label"] = labels
    return df

In [17]:
def make_windows_train(values: np.ndarray, labels: np.ndarray, seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values).reshape(-1, 1)
    labels = np.asarray(labels)

    n_windows = len(values) - seq_len + 1
    if n_windows <= 0:
        raise ValueError("Sequence length is greater than the total series length.")

    x_windows = np.empty((n_windows, seq_len, 1), dtype=np.float32)
    y_windows = np.empty((n_windows,), dtype=np.float32)

    for i in range(n_windows):
        x_windows[i] = values[i : i + seq_len]
        y_windows[i] = np.max(labels[i : i + seq_len])

    return x_windows, y_windows


def train_model(
    model: torch.nn.Module,
    train_x: np.ndarray,
    lr: float = 1e-3,
    epochs: int = 10,
    batch_size: int = 64,
    device: str = "cpu",
) -> torch.nn.Module:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # ---- Section Name ----
    # Initialize learning rate scheduler to decay LR when validation loss plateaus
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min", 
        factor=0.5, 
        patience=3
    )
    
    criterion = torch.nn.MSELoss()
    model.train()
    
    x = torch.tensor(train_x, dtype=torch.float32)
    if x.ndim != 3:
        raise ValueError(f"Invalid input shape: {x.shape}")

    dataset = TensorDataset(x)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=(device == "cuda"))

    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0

        for (batch,) in loader:
            batch = batch.to(device)
            pred = model(batch)

            loss = criterion(pred, batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        epoch_loss = total_loss / max(n_batches, 1)
        
        # ---- Section Name ----
        # Update lr scheduler and extract current learning rate for tracking
        scheduler.step(epoch_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(f"epoch {epoch+1}/{epochs} | loss {epoch_loss:.6f} | lr {current_lr:.6f}")

    return model

In [ ]:
def make_windows_inference(values: np.ndarray, seq_len: int) -> np.ndarray:
    values = np.asarray(values).reshape(-1, 1)
    n_windows = len(values) - seq_len + 1
    if n_windows <= 0:
        raise ValueError("Sequence length is greater than the total series length.")

    x_windows = []
    for i in range(n_windows):
        x_windows.append(values[i : i + seq_len])

    return np.stack(x_windows).astype(np.float32)


def run_inference(
    model: torch.nn.Module, values: np.ndarray, seq_len: int, device: str = "cpu", batch_size: int = 256
) -> tuple[np.ndarray, np.ndarray]:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()

    if values.ndim == 3:
        x = values
    else:
        x = make_windows_inference(values, seq_len)

    x_tensor = torch.tensor(x, dtype=torch.float32)
    dataset = TensorDataset(x_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    errors = []
    recons = []

    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            recon = model(batch)

            err = torch.mean((recon - batch) ** 2, dim=(1, 2))
            errors.append(err.cpu())
            recons.append(recon.cpu())

    return torch.cat(errors).numpy(), torch.cat(recons).numpy()


def compute_threshold(train_errors: np.ndarray, percentile: float = 95.0) -> float:
    return float(np.percentile(train_errors, percentile))


def detect_anomalies(errors: np.ndarray, threshold: float) -> np.ndarray:
    return (np.asarray(errors) > threshold).astype(np.int32)


def inference_pipeline(
    model: torch.nn.Module, train_values: np.ndarray, test_values: np.ndarray, seq_len: int
) -> tuple[np.ndarray, np.ndarray, float, np.ndarray]:
    train_err, _ = run_inference(model, train_values, seq_len)
    threshold = compute_threshold(train_err, 95.0)
    test_err, _ = run_inference(model, test_values, seq_len)
    preds = detect_anomalies(test_err, threshold)

    return train_err, test_err, threshold, preds

In [33]:
# ---- config ---- #
SEQ_LEN = 50
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")
windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv",
)
df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)

# ---- windowing ---- #
x, y = make_windows_train(values, labels, SEQ_LEN) # type: ignore

# ---- TRUE ANOMALY STATS ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")

# ---- NORMAL ONLY TRAIN SET ---- #
train_x = x[y == 0]

# ---- model ---- #
model = LSTMAutoencoder(input_dim=1, hidden_dim=128, latent_dim=32)

# ---- train ---- #
model = train_model(model=model, train_x=train_x, epochs=100, batch_size=64, device=device)

# ---- save model ---- #
torch.save(model.state_dict(), "lstm_ae_model.pt")
print("model saved")

device: cuda
true_anomaly_ratio: 0.113482 (11.35%)
windows: 3983
anomaly_windows: 452
normal_windows: 3531
epoch 1/100 | loss 2112.382065 | lr 0.001000
epoch 2/100 | loss 1621.587576 | lr 0.001000
epoch 3/100 | loss 1402.550864 | lr 0.001000
epoch 4/100 | loss 1260.782911 | lr 0.001000
epoch 5/100 | loss 1052.292887 | lr 0.001000
epoch 6/100 | loss 803.557695 | lr 0.001000
epoch 7/100 | loss 644.177031 | lr 0.001000
epoch 8/100 | loss 513.279544 | lr 0.001000
epoch 9/100 | loss 424.468973 | lr 0.001000
epoch 10/100 | loss 332.934281 | lr 0.001000
epoch 11/100 | loss 282.061586 | lr 0.001000
epoch 12/100 | loss 240.457156 | lr 0.001000
epoch 13/100 | loss 193.081794 | lr 0.001000
epoch 14/100 | loss 184.243462 | lr 0.001000
epoch 15/100 | loss 144.974169 | lr 0.001000
epoch 16/100 | loss 140.205733 | lr 0.001000
epoch 17/100 | loss 127.330001 | lr 0.001000
epoch 18/100 | loss 134.554604 | lr 0.001000
epoch 19/100 | loss 120.822392 | lr 0.001000
epoch 20/100 | loss 95.523339 | lr 0.00100

In [ ]:
# ---- config ---- #
SEQ_LEN = 50
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- load data ---- #
df = load_series("../assets/data/art_daily_flatmiddle.csv")
windows = load_windows(
    json_path="../assets/labels/combined_windows.json",
    file_key="artificialWithAnomaly/art_daily_flatmiddle.csv",
)
df = build_labels(df, windows)

values = df["value"].values.astype(np.float32)
labels = df["label"].values.astype(np.float32)

# ---- windowing (GROUND TRUTH SPACE) ---- #
x, y = make_windows_train(values, labels, SEQ_LEN) # type: ignore

# ---- stats ---- #
true_anomaly_ratio = np.mean(y == 1)
print(f"true_anomaly_ratio: {true_anomaly_ratio:.6f} ({true_anomaly_ratio*100:.2f}%)")
print(f"windows: {len(y)}")
print(f"anomaly_windows: {np.sum(y == 1)}")
print(f"normal_windows: {np.sum(y == 0)}")

# ---- load model ---- #
model = LSTMAutoencoder(input_dim=1, hidden_dim=128, latent_dim=32)
model.load_state_dict(torch.load("lstm_ae_model.pt", map_location=device))
model.to(device)
model.eval()

# ---- TRAIN distribution errors (NORMAL ONLY) ---- #
train_x = x[y == 0]
train_err, _ = run_inference(model, train_x, SEQ_LEN, device=device)

# ---- TEST (FULL SERIES - FIXED) ---- #
test_err, _ = run_inference(model, values, SEQ_LEN, device=device) # type: ignore

# ---- threshold ---- #
# Updated to match project core requirement guidelines (95th percentile calibration baseline)
threshold = compute_threshold(train_err, percentile=95.0)

# ---- predictions ---- #
preds = (test_err > threshold).astype(np.int32)

# ---- output ---- #
print("threshold:", threshold)
print("train_error_mean:", train_err.mean())
print("test_error_mean:", test_err.mean())
print("anomalies:", preds.sum())
print("anomaly_ratio:", preds.mean())

device: cuda
true_anomaly_ratio: 0.113482 (11.35%)
windows: 3983
anomaly_windows: 452
normal_windows: 3531
threshold: 54.53515625
train_error_mean: 18.076128
test_error_mean: 39.94108
anomalies: 246
anomaly_ratio: 0.061762490584986195
